# Pre-class: Machine learning interatomic potentials (MLIP) basics

**MS.49900 AI for Materials Science, Hands-on session 7 (pre-class)**

Finish this notebook before class (about 60 min). In class we will compare several universal MLIPs and fine-tune one.

1. Set up Colab and SevenNet
2. ASE basics (`Atoms`, `Calculator`) and how an MLIP works
3. Practice: predict DFT energies, forces and stress of Li₆PS₅Cl with SevenNet-0
4. What the results mean (read before class)

Cells marked `# You can change here` are blanks you fill in.

## 1. Setup
Use a GPU runtime: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU. Runtime > Change runtime type > T4 GPU, then run again from the top.')

In [ ]:
# Install SevenNet and download the course repository with the Li6PS5Cl DFT data (about 1 min)
!pip install -q sevenn==0.13.0

import os
if not os.path.exists('MS49900-AI4M'):
    !git clone -q https://github.com/kwongibaek/MS49900-AI4M.git

DATA = 'MS49900-AI4M/Data'
print(sorted(f for f in os.listdir(DATA) if f.startswith(('lpsc_', 'sevennet_'))))

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ase.io import read
from ase.units import bar
from sevenn.calculator import SevenNetCalculator

to_kbar = 1 / bar / 1000          # eV/Å³ -> kbar


def dft_labels(traj):
    """DFT energy (eV/atom), forces (eV/Å) and stress (kbar) stored in the data file."""
    E = np.array([a.get_potential_energy() / len(a) for a in traj])
    F = np.concatenate([a.get_forces().ravel() for a in traj])
    S = np.concatenate([a.get_stress().ravel() for a in traj]) * to_kbar
    return E, F, S

## 2. ASE basics

[ASE](https://wiki.fysik.dtu.dk/ase/) stores a structure as an `Atoms` object.
A **calculator** attached to `Atoms` returns energy, forces and stress. DFT codes and MLIPs both work as calculators, so the same simulation code runs with either.

Our system is **Li₆PS₅Cl** (argyrodite), a solid electrolyte for all-solid-state batteries. The data are DFT MD snapshots of a 2×2×2 supercell (416 atoms) at 600 K and 1200 K.

In [ ]:
atoms = read(f'{DATA}/lpsc_600K.extxyz.gz', index=0)
print('Formula :', atoms.get_chemical_formula())
print('Atoms   :', len(atoms))
print('Volume  :', round(atoms.get_volume(), 1), 'Å³ (2x2x2 supercell)')

# The file also stores DFT results for each snapshot
print('DFT energy :', round(atoms.get_potential_energy() / len(atoms), 4), 'eV/atom')
print('DFT forces :', atoms.get_forces().shape, 'eV/Å')

In [ ]:
from ase.visualize import view
view(atoms, viewer='x3d')

### How an MLIP works

An MLIP learns the potential energy surface from DFT data and then predicts energies much faster than DFT.

1. **Total energy = sum of atomic energies**, $E = \sum_i E_i$. Each $E_i$ depends only on the neighbours of atom $i$ within a cutoff radius (5 Å for `7net-0`), so one model works for cells of any size.
2. **Forces and stress come from the same energy**: $\mathbf{F}_i = -\partial E / \partial \mathbf{r}_i$ and $\sigma = \frac{1}{V}\,\partial E / \partial \varepsilon$ ($\varepsilon$: strain). The model gets them by automatic differentiation, so they are consistent with the energy.
3. **SevenNet** is a graph neural network. Atoms are nodes and pairs closer than the cutoff are edges. Five message-passing layers update each atom's features from its neighbours, respecting rotations (equivariant, as in NequIP).
4. **Element reference energies.** The model scales each atomic output and adds a reference energy for the element, $E_i = s\,\hat{E}_i + \mu_{Z_i}$. Changing every $\mu$ by the same amount shifts the energy per atom by a constant and leaves forces and stress unchanged. Keep this in mind for Section 4.

Attach a pretrained SevenNet model as the calculator. `7net-0` was trained on the Materials Project relaxation data (MPtrj).

In [ ]:
calc = SevenNetCalculator('7net-0')

mlip_atoms = atoms.copy()      # copy without the DFT results
mlip_atoms.calc = calc
print('7net-0 energy :', round(mlip_atoms.get_potential_energy() / len(mlip_atoms), 4), 'eV/atom')
print('7net-0 forces :', mlip_atoms.get_forces().shape)
print('7net-0 stress :', (mlip_atoms.get_stress() * to_kbar).round(2), 'kbar')

## 3. Practice: SevenNet-0 vs DFT

Predict all 85 snapshots at 600 K with 7net-0 and compare with DFT.

In [ ]:
traj = read(f'{DATA}/lpsc_600K.extxyz.gz', ':')
print(len(traj), 'structures')

dft_E, dft_F, dft_S = dft_labels(traj)

### What is a snapshot?

The 600 K file comes from a DFT molecular dynamics (MD) run: DFT computes the forces on all atoms, the atoms move by one short time step, and this repeats. Every so often the run saves the structure together with its DFT energy, forces and stress. Each saved structure is a **snapshot**: one frozen frame of the run.

- All 85 snapshots have the same 416-atom cell (2×2×2 unit cells, 965 Å³ per unit cell) and are in time order. Only the atomic positions differ.
- 7net-0 does **not** run MD here. It computes the energy, forces and stress of each frozen snapshot once (a *single-point* calculation), and we compare them with the stored DFT values.

First watch the whole run as a movie, then look at a few single snapshots.

**The MD run as a movie.** The next cell plays the 85 snapshots in order (drag to rotate, scroll to zoom). Purple: Li, orange: P, yellow: S, green: Cl. The gray box is the simulation cell.

- The six Li ions that moved farthest are drawn as large **red** spheres, with their paths as red lines. Watch them move from site to site, while P, S and Cl only shake in place.
- An atom that crosses the cell boundary is drawn continuing outside the box. In the simulation it re-enters from the opposite side (periodic boundary conditions).
- The movie shows only the saved snapshots, not every MD step, so the motion looks jerky.

In [ ]:
# Watch the DFT MD run as a 3D movie (drag to rotate, scroll to zoom)
import io, itertools
from ase import Atoms
from ase.io import write

symbols = np.array(traj[0].get_chemical_symbols())
cell = traj[0].cell.array
pos = np.array([a.get_positions() for a in traj])                      # snapshots x atoms x 3 (Å), wrapped into the cell

# Follow each atom across the periodic boundary so it does not jump to the other side of the box
step = np.diff(pos, axis=0) @ np.linalg.inv(cell)
step -= np.round(step)
unwrapped = pos[0] + np.concatenate([np.zeros((1, *pos.shape[1:])), np.cumsum(step @ cell, axis=0)])

N_SHOW = 6                                                             # Li ions to highlight
li = np.where(symbols == 'Li')[0]
moved = np.linalg.norm(unwrapped[-1, li] - unwrapped[0, li], axis=1)
fast = li[np.argsort(moved)[-N_SHOW:]]                                 # the Li that moved farthest

!pip install -q py3Dmol
import py3Dmol

movie = [Atoms(symbols, positions=p) for p in unwrapped]
buf = io.StringIO(); write(buf, movie, format='xyz')

v = py3Dmol.view(width=700, height=500)
v.addModelsAsFrames(buf.getvalue(), 'xyz')
v.setStyle({'sphere': {'scale': 0.2, 'colorscheme': 'Jmol'}})
v.setStyle({'index': fast.tolist()}, {'sphere': {'scale': 0.45, 'color': 'red'}})
for i in fast:                                                         # path of each highlighted Li
    for p, q in zip(unwrapped[:-1, i], unwrapped[1:, i]):
        v.addCylinder({'start': dict(zip('xyz', p.tolist())), 'end': dict(zip('xyz', q.tolist())),
                       'radius': 0.07, 'color': 'red', 'fromCap': 1, 'toCap': 1})
corners = np.array(list(itertools.product([0, 1], repeat=3))) @ cell    # outline of the simulation cell
edge = np.linalg.norm(cell, axis=1)
for a, b in itertools.combinations(corners, 2):
    if np.isclose(np.linalg.norm(a - b), edge).any():
        v.addCylinder({'start': dict(zip('xyz', a.tolist())), 'end': dict(zip('xyz', b.tolist())),
                       'radius': 0.05, 'color': 'gray'})
v.zoomTo(); v.rotate(-25, 'x'); v.rotate(30, 'y')
v.animate({'loop': 'forward', 'interval': 120}); v.show()

**Single snapshots.** The next cell freezes four frames of the movie (snapshots 0, 28, 56 and 84). To see the structure clearly it draws only one unit cell (52 atoms) and follows the same atoms in every frame. Colours as in the movie; red: the two Li ions in this unit cell that move farthest. Each title gives the DFT energy and the largest force on an atom in that snapshot.

In [ ]:
# Four single snapshots: frozen frames of the movie above.
# For a clear view we draw one unit cell (1/8 of the simulation cell, 52 atoms) and follow the same atoms.
from ase.visualize.plot import plot_atoms
from ase.data import atomic_numbers
from ase.data.colors import jmol_colors

SHOW = [0, 28, 56, 84]                                                 # snapshot indices
frac0 = pos[0] @ np.linalg.inv(cell)
sel = np.where((frac0 < 0.5).all(axis=1))[0]                           # atoms that start in one unit cell
sel_li = sel[symbols[sel] == 'Li']
mark = sel_li[np.argsort(np.linalg.norm(unwrapped[-1, sel_li] - unwrapped[0, sel_li], axis=1))[-2:]]  # 2 Li that move most

colors = ['red' if i in mark else jmol_colors[atomic_numbers[symbols[i]]] for i in sel]
radii = [0.6 if i in mark else 0.5 for i in sel]

fig, axes = plt.subplots(1, len(SHOW), figsize=(4.3 * len(SHOW), 4.8))
for ax, k in zip(axes, SHOW):
    unit = Atoms(symbols[sel], positions=unwrapped[k, sel], cell=cell / 2, pbc=True)
    plot_atoms(unit, ax, rotation='-25x,30y,0z', colors=colors, radii=radii)
    E = traj[k].get_potential_energy() / len(traj[k])
    F_max = np.linalg.norm(traj[k].get_forces(), axis=1).max()
    ax.set_title(f'Snapshot {k}\nDFT: E = {E:.3f} eV/atom, max |F| = {F_max:.1f} eV/Å', fontsize=10)
    ax.axis('off')
plt.tight_layout(); plt.show()

**What to see.**
- Each panel is one complete structure with its own DFT labels. These frozen structures are what we give to 7net-0, one at a time, in Blank 1 below.
- P, S and Cl stay near the same positions in all four frames: they only vibrate, so the PS₄ / S / Cl framework does not change.
- The red Li ions are at different places in different frames: Li ions hop between sites. This is why Li₆PS₅Cl conducts Li ions so well; we measure the hopping in class and in the post-class notebook.
- The DFT energy changes only in the second decimal place (about 0.03 eV/atom between these four snapshots). These small differences are what the model must get right, which is why the energy parity plot below uses relative energies.

**Blank 1.** Fill the prediction loop.

In [ ]:
E, F, S = [], [], []
for frame in traj:
    snap = frame.copy()       # copy without the DFT results
    snap.calc = calc
# You can change here------------------------------------------------------------
    E.append(...)   # energy per atom (eV/atom)
    F.append(...)   # forces, flattened (eV/Å)
    S.append(...)   # stress in kbar (multiply by to_kbar)
# -------------------------------------------------------------------------------

mlip_E, mlip_F, mlip_S = np.array(E), np.concatenate(F), np.concatenate(S)
print(len(mlip_E), 'energies,', len(mlip_F), 'force components,', len(mlip_S), 'stress components')

**Blank 2.** Write RMSE and MAE.

In [ ]:
# You can change here------------------------------------------------------------
def rmse(y_true, y_pred):
    return ...   # root mean squared error

def mae(y_true, y_pred):
    return ...   # mean absolute error
# -------------------------------------------------------------------------------

table = pd.DataFrame({
    'RMSE': [rmse(dft_E, mlip_E), rmse(dft_F, mlip_F), rmse(dft_S, mlip_S)],
    'MAE':  [mae(dft_E, mlip_E),  mae(dft_F, mlip_F),  mae(dft_S, mlip_S)],
}, index=['Energy (eV/atom)', 'Force (eV/Å)', 'Stress (kbar)'])
table.round(4)

### Parity plots

7net-0 energies are shifted by a constant (see Section 4), so absolute energies would all sit far from the diagonal. The energy plot therefore uses **relative energies** (each set minus its own mean) to show whether the model ranks the snapshots correctly.

In [ ]:
def parity(ax, x, y, label, unit):
    ax.scatter(x, y, s=3, alpha=0.3)
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
    ax.set_xlabel(f'DFT {label} ({unit})')
    ax.set_ylabel(f'MLIP {label} ({unit})')
    ax.set_aspect('equal')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
parity(axes[0], dft_E - dft_E.mean(), mlip_E - mlip_E.mean(), 'relative energy', 'eV/atom')
parity(axes[1], dft_F, mlip_F, 'force', 'eV/Å')
parity(axes[2], dft_S, mlip_S, 'stress', 'kbar')
fig.suptitle('7net-0 vs DFT, Li₆PS₅Cl 600 K')
plt.tight_layout(); plt.show()

print('Mean energy  DFT:', dft_E.mean().round(3), ' 7net-0:', mlip_E.mean().round(3), 'eV/atom')

## 4. What the results mean (read before class)

### What you see
- **Forces and stress** lie close to the diagonal. 7net-0 has never seen this dataset, yet it already gets forces and stress about right. This is what a *universal* MLIP promises: it works zero-shot on a new material.
- **Energy** looks bad: the RMSE is about 2.4 eV/atom. But three results show what kind of error it is:
  1. The energy RMSE and MAE are almost equal. For random errors the RMSE is clearly larger than the MAE (about 1.25× for a normal distribution); they are equal only when every structure has the same error.
  2. The relative-energy parity plot follows the diagonal, so 7net-0 ranks the snapshots correctly.
  3. The two mean energies differ by about the same 2.4 eV/atom.

  So the energy error is a **constant offset**, not noise.

### Why the offset appears
- The absolute DFT energy is not a physical observable. It depends on the calculation setting (code, pseudopotentials, functional, reference energies). Only energy **differences** between structures computed with the same setting are meaningful.
- An MLIP reproduces the DFT setting of its training data. 7net-0 learned Materials Project energies (VASP, PBE/PBE+U), while this dataset was computed with a different setting, so every 7net-0 energy is shifted.
- In the model this is a difference in the element reference energies $\mu_Z$ (Section 2). Our snapshots all have the same composition, so the four element shifts add up to one constant per atom. Forces and stress are derivatives of the energy, so the constant drops out of them.

### When the offset is a problem
- **Not for MD of this material.** MD only uses forces, so the dynamics are unaffected.
- **When you compare energies from different sources**: for example 7net-0 energies with your own DFT energies, or a reaction energy that mixes both.
- **When the composition changes**: each element has its own shift, so a single constant no longer removes the error. Formation energies and phase stability are affected.

So a large RMSE does not always mean a bad model. Before trusting or rejecting a number, check *what kind* of error it is.

### Bring to class
- How an MLIP builds the energy from atomic contributions and element reference energies (Section 2).
- What RMSE and MAE each measure, and what their ratio tells you.
- A universal MLIP reproduces the DFT setting of its training data. Multi-task models (`7net-mf-ompa`, `7net-omni`) let you choose the setting with `modal` (see the table below).

In class we start from this offset: we compare several universal MLIPs after removing it, remove it from a small model by shifting its reference energies without any training, and then fine-tune the model to improve forces and stress as well.

## Optional: other SevenNet pretrained models

`SevenNetCalculator` accepts other pretrained models by name. Some multi-task models need `modal` (which DFT setting to reproduce).

| Name | Training data | Note |
|---|---|---|
| `7net-0` | MPtrj | small and fast |
| `7net-l3i5` | MPtrj | higher order, slower |
| `7net-mf-ompa` | MPtrj + sAlex + OMat24 | `modal='mpa'` or `'omat24'` |
| `7net-omat` | OMat24 | |
| `7net-omni` | 15 open DFT datasets | `modal='mpa'`, `'matpes_pbe'`, ... |

The next cell predicts the same 85 snapshots with each model and shows the same table values and parity plots as for 7net-0. Compare the mean energy with the DFT mean: does a bigger model or more training data remove the offset, and does it improve forces and stress? In class we compare some of these models on a separate test set.

In [ ]:
# Optional: predict the same 85 snapshots with the other pretrained models and compare with 7net-0.
# Takes a few minutes on a T4; each model is downloaded the first time it is used.
import gc, torch

OTHER_MODELS = {                      # label: (model name, extra options)
    '7net-l3i5':    ('7net-l3i5', {}),
    '7net-omat':    ('7net-omat', {}),
    '7net-mf-ompa': ('7net-mf-ompa', {'modal': 'mpa'}),
    '7net-omni':    ('7net-omni', {'modal': 'mpa'}),
}

results = {'7net-0': (mlip_E, mlip_F, mlip_S)}
for label, (name, options) in OTHER_MODELS.items():
    model = SevenNetCalculator(name, **options)
    E_m, F_m, S_m = [], [], []
    for snap in traj:
        a = snap.copy(); a.calc = model
        E_m.append(a.get_potential_energy() / len(a))
        F_m.append(a.get_forces().ravel())
        S_m.append(a.get_stress().ravel() * to_kbar)
    results[label] = (np.array(E_m), np.concatenate(F_m), np.concatenate(S_m))
    del model, a; gc.collect()                                     # free GPU memory before the next model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(label, 'done')

summary = pd.DataFrame({label: {
    'Mean energy (eV/atom)': E_m.mean(),
    'Energy RMSE (eV/atom)': rmse(dft_E, E_m),
    'Force RMSE (eV/Å)': rmse(dft_F, F_m),
    'Stress RMSE (kbar)': rmse(dft_S, S_m),
} for label, (E_m, F_m, S_m) in results.items()}).T
print(f'Mean DFT energy: {dft_E.mean():.3f} eV/atom')
display(summary.round(3))

fig, axes = plt.subplots(len(results), 3, figsize=(12, 3.8 * len(results)))
for row, (label, (E_m, F_m, S_m)) in zip(axes, results.items()):
    parity(row[0], dft_E - dft_E.mean(), E_m - E_m.mean(), 'relative energy', 'eV/atom')
    parity(row[1], dft_F, F_m, 'force', 'eV/Å')
    parity(row[2], dft_S, S_m, 'stress', 'kbar')
    row[0].set_title(label, loc='left', fontweight='bold')
plt.tight_layout(); plt.show()